# Turning a Photo into a Pencil Sketch with OpenCV

For this exercise, I wanted to see how a regular colour photograph could be turned into something that looks like a pencil drawing using a few familiar image processing steps.

The method is built around a simple idea: remove the colour, invert the image, soften that negative with a Gaussian blur, and blend it back with the grayscale original. The result is a sketch effect, and changing the blur strength gives us a few different looks to compare.

### What we'll do

1. Load a colour image and convert it to RGB.
2. Turn it into grayscale and make an inverted copy.
3. Blur the inverted image, then invert it again.
4. Use a dodge blend to create the pencil sketch.
5. Try a few blur settings and an optional contrast adjustment.
6. Save both versions of the result.

**Getting an image into the notebook:** In Google Colab, the upload window will appear when you run the loading cell. In a local Jupyter setup, put a JPG or PNG in the same folder as this notebook and set its name in `IMAGE_PATH`.

## 1. Get the libraries ready

We'll use OpenCV for the image processing, NumPy for working with pixel values, and Matplotlib to see what each step is doing.

The install command is here in case you need it. If these libraries are already installed, you can skip straight to the imports.

In [ ]:
# Run this only if the libraries used below are not installed.
# %pip install opencv-python numpy matplotlib

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

## 2. Load the image

Let's start with the photograph we want to turn into a sketch.

One small OpenCV detail is worth keeping in mind: `imread()` loads colour images in BGR order, not RGB. I'll convert it straight away so the rest of the notebook can work with a standard RGB image.

The cell checks whether it is running in Google Colab. If it is, you'll be asked to upload a file. Otherwise, it looks for `input_image.jpg` in the notebook's folder.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No image was uploaded.")
    IMAGE_PATH = next(iter(uploaded))
else:
    IMAGE_PATH = "input_image.jpg"  # Replace this with the name of your image.

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)

if image_bgr is None:
    raise FileNotFoundError(
        f"Could not read '{IMAGE_PATH}'. Check the filename and make sure it is a valid image."
    )

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print("Image path:", IMAGE_PATH)
print("Image dimensions (height, width, channels):", image_rgb.shape)
print("Data type:", image_rgb.dtype)
print("Pixel value range:", int(image_rgb.min()), "to", int(image_rgb.max()))

In [ ]:
def show_image(image, title, cmap=None, figsize=(8, 6)):
    """Show an image clearly, without the plot axes around it."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap=cmap or "gray", vmin=0, vmax=255)
    else:
        plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

show_image(image_rgb, "Original RGB image")

## 3. Remove the colour

A pencil drawing is usually made up of light and shade rather than colour, so the first transformation is to convert the photograph to grayscale.

Because the image is now in RGB order, we use `COLOR_RGB2GRAY`. Each pixel is reduced to a single brightness value, which gives us the starting point for the sketch.

In [ ]:
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

print("Grayscale image shape:", gray.shape)
show_image(gray, "Grayscale image")

## 4. Make a negative

Next, I'll invert the grayscale image. Bright areas become dark and dark areas become bright.

This negative image is not the sketch yet. It gives us something to blur in the next step, which is what helps create the softer shading in the final result.

In [ ]:
inverted_gray = cv2.bitwise_not(gray)
show_image(inverted_gray, "Inverted grayscale image")

## 5. Soften it with a Gaussian blur

Now we apply a Gaussian blur to the inverted image. In simple terms, this softens the sharp changes between neighbouring pixels.

The amount of blur matters. A smaller kernel usually keeps more detail, while a larger one tends to produce a softer, lighter looking sketch. I'll start with a kernel size of `21 × 21` and compare a few alternatives later.

OpenCV expects the kernel dimensions to be odd numbers, so use values such as 11, 21 or 31.

In [ ]:
BLUR_KERNEL_SIZE = 21  # Try 11, 21 or 31 to compare the amount of blur.
BLUR_SIGMA = 0          # Let OpenCV choose sigma from the kernel size.

if BLUR_KERNEL_SIZE <= 0 or BLUR_KERNEL_SIZE % 2 == 0:
    raise ValueError("BLUR_KERNEL_SIZE must be a positive odd number.")

blurred = cv2.GaussianBlur(
    inverted_gray,
    (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE),
    sigmaX=BLUR_SIGMA
)

show_image(blurred, f"Gaussian blurred inverted image, kernel {BLUR_KERNEL_SIZE} x {BLUR_KERNEL_SIZE}")

## 6. Invert the blurred image

The blurred image is inverted again. This gives us the image that will be used as the divisor in the next step.

We're nearly at the sketch effect now. The final blend combines this softened image with the original grayscale photograph.

In [ ]:
inverted_blurred = cv2.bitwise_not(blurred)
show_image(inverted_blurred, "Inverted blurred image")

## 7. Blend the images to create the sketch

Here is the key step. We divide the original grayscale image by the inverted blurred image. This is often called a *colour dodge* or *dodge blend*.

The division brightens the smoother, lighter areas while allowing darker outlines and details to remain visible. OpenCV's `cv2.divide()` does the pixel by pixel calculation for us. The scale value of `256` is used to bring out the pencil sketch effect.

In [ ]:
pencil_sketch = cv2.divide(gray, inverted_blurred, scale=256.0)
show_image(pencil_sketch, "Pencil sketch")

## 8. See what changing the blur does

A single blur setting won't suit every photograph, so let's compare three of them.

The images below use the same process with kernel sizes of 11, 21 and 31. Look at how much detail is kept around edges and textures, and how the lighter areas change as the blur gets stronger.

In [ ]:
kernel_sizes = [11, 21, 31]

fig, axes = plt.subplots(1, len(kernel_sizes), figsize=(18, 6))

for ax, kernel_size in zip(axes, kernel_sizes):
    inverted = cv2.bitwise_not(gray)
    blur = cv2.GaussianBlur(inverted, (kernel_size, kernel_size), sigmaX=0)
    inverted_blur = cv2.bitwise_not(blur)
    sketch = cv2.divide(gray, inverted_blur, scale=256.0)

    ax.imshow(sketch, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Kernel size: {kernel_size} x {kernel_size}")
    ax.axis("off")

plt.suptitle("Effect of Gaussian blur strength", fontsize=15)
plt.tight_layout()
plt.show()

## 9. An optional contrast adjustment

Depending on the photograph, the basic sketch can sometimes look a little flat. One way to bring out local detail is CLAHE, short for Contrast Limited Adaptive Histogram Equalization.

CLAHE adjusts contrast in small areas of the grayscale image rather than applying one change to the whole picture. I'll run the same sketch process again using this adjusted grayscale version, then place the two results side by side.

This is optional, so compare the results. More contrast is not automatically better if it makes the sketch look harsh.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced_gray = clahe.apply(gray)

enhanced_inverted = cv2.bitwise_not(enhanced_gray)
enhanced_blurred = cv2.GaussianBlur(
    enhanced_inverted,
    (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE),
    sigmaX=BLUR_SIGMA
)
enhanced_inverted_blurred = cv2.bitwise_not(enhanced_blurred)

enhanced_sketch = cv2.divide(
    enhanced_gray,
    enhanced_inverted_blurred,
    scale=256.0
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original grayscale")
axes[1].imshow(enhanced_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("CLAHE enhanced grayscale")
axes[2].imshow(enhanced_sketch, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Enhanced pencil sketch")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 10. Save the sketches

Once you're happy with the result, this cell saves both versions as PNG files: the standard pencil sketch and the version made with CLAHE.

The files go into a folder called `output`, which the notebook creates if it doesn't already exist.

In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

basic_output_path = output_dir / "pencil_sketch.png"
enhanced_output_path = output_dir / "pencil_sketch_enhanced.png"

if not cv2.imwrite(str(basic_output_path), pencil_sketch):
    raise IOError(f"Could not save {basic_output_path}")

if not cv2.imwrite(str(enhanced_output_path), enhanced_sketch):
    raise IOError(f"Could not save {enhanced_output_path}")

print("Saved:", basic_output_path.resolve())
print("Saved:", enhanced_output_path.resolve())

## 11. A few final thoughts

The pencil sketch effect comes from combining a handful of simple operations: grayscale conversion, inversion, Gaussian blur and a dodge blend. The interesting part is how much the result can change when the blur strength changes.

The CLAHE version gives us another option when the original photo needs a little more local contrast, but it won't improve every image. Lighting, background detail and texture all affect the result.

A good way to get a feel for the technique is to try a few different photographs. A portrait, an animal photo and an image with strong shapes or edges will each respond a little differently. Compare the results and see which settings work best for each one.